In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import importlib
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

import ev_robust_charging as ev

# Pick up local edits in ev_robust_charging.py without restarting the kernel.
ev = importlib.reload(ev)

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

DELTA = 0.5
FORCE_RECOMPUTE = False  # set True to regenerate CSVs from solvers

POLICY_ORDER = ["deterministic", "quantile_75", "robust"]
POLICY_LABELS = {
    "deterministic": "Deterministic",
    "quantile_75": "Slotwise Q75",
    "robust": "Robust",
}
COLORS = {
    "deterministic": "#2B5C8A",
    "quantile_75": "#D8892B",
    "robust": "#3A7D44",
    "empirical_90": "#7E57A0",
    "empirical_95": "#B45F4D",
    "conformal_95": "#4C7C7D",
}
MARKERS = {
    "deterministic": "o",
    "quantile_75": "s",
    "robust": "^",
    "empirical_90": "D",
    "empirical_95": "P",
    "conformal_95": "X",
}

plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 10.5,
    "legend.fontsize": 8.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.grid": True,
    "grid.alpha": 0.28,
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "figure.dpi": 120,
})

In [ ]:
def optimal_mask(frame: pd.DataFrame, status_col: str = "status") -> pd.Series:
    return frame[status_col].isin(["optimal", "optimal_inaccurate"])


def save_table(frame: pd.DataFrame, filename: str) -> pd.DataFrame:
    path = OUT / filename
    frame.to_csv(path, index=False)
    print(f"Saved {path}")
    display(frame)
    return frame


def show_figure(path: str | Path, width: int = 900) -> None:
    display(Image(filename=str(OUT / path), width=width))


def save_figure(fig, stem: str, width: int = 900) -> None:
    png_path = OUT / f"{stem}.png"
    pdf_path = OUT / f"{stem}.pdf"
    fig.savefig(png_path, dpi=300)
    fig.savefig(pdf_path)
    print(f"Saved {png_path} and {pdf_path}")
    plt.close(fig)
    show_figure(png_path.name, width=width)


def format_residual(value: float, tol: float = 1.0e-8) -> str:
    value = float(abs(value))
    return rf"$<{tol:.0e}$" if value <= tol else f"{value:.1e}"


def protected_policy(policy_name: str) -> ev.Policy:
    return next(policy for policy in ev.POLICIES if policy.name == policy_name)


def solve_base_policy_solutions(base: ev.EVData, delta: float = DELTA) -> dict[str, dict]:
    return {
        policy.name: ev.solve_model(base, delta=delta, policy=policy)
        for policy in ev.POLICIES
        if policy.name in POLICY_ORDER
    }


def ensure_station_outputs(base: ev.EVData) -> pd.DataFrame:
    path = OUT / "q1_baseline_policy_rerun.csv"
    if FORCE_RECOMPUTE or not path.exists():
        station_summary, _ = ev.run_delta_sensitivity(base, list(ev.SENSITIVITY_DELTAS))
        station_summary.to_csv(OUT / "summary.csv", index=False)
        station_summary.to_csv(path, index=False)
    return pd.read_csv(path)


def ensure_multi_day_outputs() -> pd.DataFrame:
    path = OUT / "multi_day_benchmark_delta_50.csv"
    if FORCE_RECOMPUTE or not path.exists():
        multi_day = ev.run_multi_day_benchmark(None, delta=DELTA)
        multi_day.to_csv(path, index=False)
    return pd.read_csv(path)


def ensure_binary_outputs(base: ev.EVData) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw_path = OUT / "q1_binary_complementarity_validation.csv"
    summary_path = OUT / "q1_binary_complementarity_summary.csv"
    if FORCE_RECOMPUTE or not raw_path.exists() or not summary_path.exists():
        raw = ev.run_binary_validation(base, delta=DELTA)
        raw.to_csv(raw_path, index=False)
        summary = ev.summarize_binary_validation(raw)
        summary.to_csv(summary_path, index=False)
    return pd.read_csv(raw_path), pd.read_csv(summary_path)

## Build and Load Single-EV Outputs

In [ ]:
base = ev.build_base_case()
station_summary = ensure_station_outputs(base)
multi_day = ensure_multi_day_outputs()
binary_raw, binary_summary = ensure_binary_outputs(base)
base_solutions = solve_base_policy_solutions(base, delta=DELTA)

print("Station rows:", station_summary.shape)
print("Multi-day rows:", multi_day.shape)
print("MIQP comparisons:", binary_summary.shape)
print("Base policy statuses:", {k: v["status"] for k, v in base_solutions.items()})

# Tables

In [ ]:
# Table 4: Experimental setup and evaluation protocol.
# The code module keeps the legacy names X and Y, but the paper interprets them as
# average charging/discharging powers P^c and P^d over one-hour slots.
table4 = pd.DataFrame([
    ("Horizon", r"24 hourly slots; $\Delta t=1$ h."),
    ("OPSD profile", "DE-LU load and day-ahead price; load scaled to a station-level stress-test profile."),
    ("Single-EV setting", rf"Availability hours 11--21; $P^c={base.X:g}$ kW, $P^d={base.Y:g}$ kW; $E_0={base.E0:g}$ kWh, $E^{{\rm tar}}={base.Etarget:g}$ kWh; bounds $[{base.Emin:g},{base.Emax:g}]$ kWh."),
    ("Reliability model", r"Base case $\delta=0.5$ with interval grid-impact uncertainty."),
    ("Policies", "Deterministic, Slotwise Q75, Robust, Empirical, and Conformal for station-level tests."),
    ("Solvers", "CVXPY with OSQP/CLARABEL for convex QPs; GUROBI for explicit MIQP validation."),
    ("Metrics", "EUR-equivalent cost, V2G energy, violation probability, worst-case excess, and complementarity residual."),
], columns=["Component", "Setting"])
save_table(table4, "table4_experimental_setup_protocol.csv")

In [ ]:
# Table 5: Exactness diagnostics: relaxed convex QP versus explicit MIQP.
objective_gap = binary_summary["objective_gap_vs_relaxation"].abs()
charge_gap = binary_summary["charge_l1_gap_vs_relaxation"].abs()
discharge_gap = binary_summary["discharge_l1_gap_vs_relaxation"].abs()
relaxed_comp = binary_summary["relaxed_max_simultaneous_flow"].abs()
miqp_comp = binary_summary["miqp_max_simultaneous_flow"].abs()

table5 = pd.DataFrame([
    ("Number of comparisons", "MIQP--relaxation pairs over tested subinstances and policies", f"{len(binary_summary)}"),
    ("MIQP status", "Solver status for explicit complementarity formulation", "all optimal" if (binary_summary["miqp_status"] == "optimal").all() else ", ".join(sorted(binary_summary["miqp_status"].unique()))),
    ("Objective gap", r"$|F_{\mathrm{MIQP}}-F_{\mathrm{rel}}|$", f"{objective_gap.max():.2e}"),
    ("Charging-profile gap", r"$\|p^{c,\mathrm{MIQP}}-p^{c,\mathrm{rel}}\|_1$", f"{charge_gap.max():.2e}"),
    ("Discharging-profile gap", r"$\|p^{d,\mathrm{MIQP}}-p^{d,\mathrm{rel}}\|_1$", f"{discharge_gap.max():.2e}"),
    ("Relaxed complementarity residual", r"$\max_t \min\{p_t^{c,\mathrm{rel}},p_t^{d,\mathrm{rel}}\}$", format_residual(relaxed_comp.max())),
    ("MIQP complementarity residual", r"$\max_t \min\{p_t^{c,\mathrm{MIQP}},p_t^{d,\mathrm{MIQP}}\}$", format_residual(miqp_comp.max())),
], columns=["Diagnostic", "Definition", "Worst value"])
save_table(table5, "table5_exactness_diagnostics.csv")

In [ ]:
# Table 6: Base-case station-level cost--reliability comparison at delta=0.5.
base_delta = station_summary[np.isclose(station_summary["delta"], DELTA)].copy()
base_delta = base_delta[base_delta["policy"].isin(POLICY_ORDER)]
base_delta["policy"] = pd.Categorical(base_delta["policy"], POLICY_ORDER, ordered=True)
base_delta = base_delta.sort_values("policy")

table6 = pd.DataFrame({
    "Policy": base_delta["policy_label"].to_numpy(),
    "EUR-eq. cost": base_delta["economic_cost"].round(3).to_numpy(),
    "V2G energy (kWh)": base_delta["total_discharge"].round(2).to_numpy(),
    "Exact violation prob.": base_delta["exact_violation_probability"].round(3).to_numpy(),
    "WC excess": base_delta["worst_case_margin"].clip(lower=0).round(2).to_numpy(),
    "Comp. residual": base_delta["max_simultaneous_flow"].map(format_residual).to_numpy(),
})
save_table(table6, "table6_base_case_station_reliability.csv")

In [ ]:
# Table 7: Multi-day and held-out reliability benchmark.
def build_proxy_coefficient_paths(base: ev.EVData) -> tuple[list[str], np.ndarray]:
    cases = ev.build_multi_day_cases(None)
    dates = [case.profile_date for case in cases]
    load_matrix = np.vstack([case.L for case in cases])
    ranks = pd.DataFrame(load_matrix).rank(method="average", pct=True).to_numpy()
    u_paths = base.u_nominal.reshape(1, base.T) * (1.0 + 0.5 * ranks)
    return dates, u_paths


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    scores = np.sort(np.asarray(scores, dtype=float))
    n = len(scores)
    rank = int(math.ceil((n + 1) * (1.0 - alpha)))
    rank = min(max(rank, 1), n)
    return float(scores[rank - 1])


def solve_calibrated_policy(base: ev.EVData, label: str, coeff: np.ndarray) -> dict:
    # Setting delta=0 and using the deterministic policy makes coeff the exact protected coefficient.
    data = replace(base, name=f"calibrated_{label}", u_nominal=np.asarray(coeff, dtype=float))
    policy = ev.Policy("deterministic", label)
    return ev.solve_model(data, delta=0.0, policy=policy)


def heldout_metrics(base: ev.EVData, solution: dict, test_paths: np.ndarray) -> tuple[float, float]:
    charging = solution["x"]
    excess = base.L.reshape(1, base.T) + test_paths * charging.reshape(1, base.T) - base.M
    path_excess = excess.max(axis=1)
    return float(np.mean(path_excess > 1.0e-6)), float(np.max(np.maximum(path_excess, 0.0)))


# Block A: multi-day OPSD benchmark.
multi_complete = multi_day[optimal_mask(multi_day)].copy()
multi_complete = multi_complete[multi_complete["policy"].isin(POLICY_ORDER)]
multi_group = (
    multi_complete.groupby(["policy", "policy_label"], as_index=False)
    .agg(
        mean_cost=("economic_cost", "mean"),
        mean_violation=("exact_violation_probability", "mean"),
        max_violation=("exact_violation_probability", "max"),
    )
)
multi_group["policy"] = pd.Categorical(multi_group["policy"], POLICY_ORDER, ordered=True)
multi_group = multi_group.sort_values("policy")
multi_table = pd.DataFrame({
    "Block": "A. Multi-day OPSD benchmark",
    "Policy": multi_group["policy_label"],
    "Cost": multi_group["mean_cost"].round(3),
    "Violation prob.": multi_group["mean_violation"].round(3),
    "Auxiliary metric": "Max violation prob.",
    "Auxiliary value": multi_group["max_violation"].round(3),
})

# Block B: held-out calibration benchmark.
dates, u_paths = build_proxy_coefficient_paths(base)
cal_paths = u_paths[:240]
test_paths = u_paths[240:]
hat = base.u_nominal
robust_coeff = 1.5 * hat
scale = 0.5 * hat
scores = np.max((cal_paths - hat.reshape(1, base.T)) / scale.reshape(1, base.T), axis=1)
conf_q = conformal_quantile(scores, alpha=0.05)
calibration_coeffs = [
    ("Deterministic", hat),
    ("Empirical 90%", np.quantile(cal_paths, 0.90, axis=0)),
    ("Empirical 95%", np.quantile(cal_paths, 0.95, axis=0)),
    ("Conformal 95%", hat + conf_q * scale),
    ("Robust", robust_coeff),
]
held_rows = []
for label, coeff in calibration_coeffs:
    sol = solve_calibrated_policy(base, label, coeff)
    violation, max_excess = heldout_metrics(base, sol, test_paths)
    held_rows.append({
        "Block": "B. Held-out calibration benchmark",
        "Policy": label,
        "Cost": round(sol["economic_cost"], 3),
        "Violation": round(violation, 3),
        "Max excess": round(max_excess, 2),
        "V2G energy": round(sol["total_discharge"], 2),
    })
heldout_table = pd.DataFrame(held_rows)

heldout_display = pd.DataFrame({
    "Block": heldout_table["Block"],
    "Policy": heldout_table["Policy"],
    "Cost": heldout_table["Cost"],
    "Violation prob.": heldout_table["Violation"],
    "Auxiliary metric": "Max excess",
    "Auxiliary value": heldout_table["Max excess"],
})

table7 = pd.concat([multi_table, heldout_display], ignore_index=True)
save_table(table7, "table7_multiday_heldout_calibration.csv")

# Keep detailed held-out data for Figure 2.
heldout_table.to_csv(OUT / "heldout_calibration_summary.csv", index=False)

# Figures

In [ ]:
# Figure 1: Station-level cost--reliability sensitivity to delta.
def plot_figure2_delta(station_summary: pd.DataFrame) -> None:
    summary = station_summary[station_summary["policy"].isin(POLICY_ORDER)].copy()
    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.4), constrained_layout=True)
    metrics = [
        ("economic_cost", "A. EUR-equivalent cost", "Net EUR-equivalent cost"),
        ("exact_violation_probability", "B. Exact violation probability", "Exact violation probability"),
    ]
    for ax, (column, title, ylabel) in zip(axes, metrics):
        for policy in POLICY_ORDER:
            frame = summary[summary["policy"] == policy].sort_values("delta")
            ax.plot(frame["delta"], frame[column], color=COLORS[policy],
                    marker=MARKERS[policy], linewidth=2.4, markersize=6.5,
                    label=POLICY_LABELS[policy])
        ax.set_title(title)
        ax.set_xlabel(r"Uncertainty radius $\delta$")
        ax.set_ylabel(ylabel)
        ax.legend(loc="best", frameon=True, framealpha=0.92, edgecolor="0.85")
        ax.grid(True, alpha=0.25)
    axes[1].set_ylim(bottom=-0.02)
    save_figure(fig, "figure1_station_delta_sensitivity", width=980)

plot_figure2_delta(station_summary)

In [ ]:
# Figure 2: Held-out calibration performance.
def plot_figure2_calibration() -> None:
    held = pd.read_csv(OUT / "heldout_calibration_summary.csv")
    policy_order = ["Deterministic", "Empirical 90%", "Empirical 95%", "Conformal 95%", "Robust"]
    color_keys = ["deterministic", "empirical_90", "empirical_95", "conformal_95", "robust"]

    held["Policy"] = pd.Categorical(held["Policy"], policy_order, ordered=True)
    held = held.sort_values("Policy")
    colors = [COLORS[key] for key in color_keys]
    x = np.arange(len(held))

    fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.0), constrained_layout=True)

    # Panel A: lollipop plot with log scale. Zero violations are drawn at a small
    # display floor so they remain visible on the logarithmic axis, but the labels
    # report the actual values.
    log_floor = 1.0e-4
    violation = held["Violation"].astype(float).to_numpy()
    violation_plot = np.maximum(violation, log_floor)
    axes[0].vlines(x, log_floor, violation_plot, color=colors, linewidth=2.2, alpha=0.8)
    axes[0].scatter(x, violation_plot, s=72, color=colors, edgecolor="white", linewidth=0.8, zorder=3)
    axes[0].set_yscale("log")
    axes[0].set_title("A. Held-out violation probability")
    axes[0].set_ylabel("Violation probability (log scale)")
    axes[0].set_xticks(x, held["Policy"], rotation=25, ha="right")
    axes[0].set_ylim(log_floor * 0.7, 1.0)
    axes[0].set_yticks([1.0e-4, 1.0e-3, 1.0e-2, 1.0e-1, 1.0])
    axes[0].axhline(log_floor, color="0.55", linewidth=0.8, linestyle=":")
    for idx, (actual, plotted) in enumerate(zip(violation, violation_plot)):
        y_text = plotted * 1.45 if plotted < 0.2 else plotted / 1.35
        va = "bottom" if plotted < 0.2 else "top"
        axes[0].text(idx, y_text, f"{actual:.3f}", ha="center", va=va, fontsize=8)

    # Panel B: cost remains on a linear scale.
    axes[1].bar(x, held["Cost"], color=colors, alpha=0.85)
    axes[1].set_title("B. EUR-equivalent cost")
    axes[1].set_ylabel("Net EUR-equivalent cost")
    axes[1].set_xticks(x, held["Policy"], rotation=25, ha="right")
    ymin = max(0.0, float(held["Cost"].min()) - 0.05)
    ymax = float(held["Cost"].max()) + 0.04
    axes[1].set_ylim(ymin, ymax)
    for idx, value in enumerate(held["Cost"]):
        axes[1].text(idx, value + 0.006, f"{value:.3f}", ha="center", va="bottom", fontsize=8)

    for ax in axes:
        ax.grid(True, axis="y", alpha=0.25, which="both")
        ax.grid(False, axis="x")
    save_figure(fig, "figure2_heldout_calibration_performance", width=920)

plot_figure2_calibration()


## Output Checklist

After running this notebook, the station-level artifacts should be regenerated in `outputs/`:

Tables:
- `table4_experimental_setup_protocol.csv`
- `table5_exactness_diagnostics.csv`
- `table6_base_case_station_reliability.csv`
- `table7_multiday_heldout_calibration.csv`

Figures:
- `figure1_station_delta_sensitivity.png/.pdf`
- `figure2_heldout_calibration_performance.png/.pdf`